In [1]:
import json
import os
import textwrap
import warnings

from dotenv import load_dotenv

warnings.filterwarnings("ignore")



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv('/Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env')  # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")


In [3]:
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )
else:
    pretty_print("API key loaded successfully.")

API key loaded successfully.


## The Story

> *Imagine you've built a customer support chatbot for **TechMart**, an online electronics store. It answers 1,000 questions a day. Your boss asks: "How good is it?"*
>
> *You check accuracy — but against what? There's no single right answer to "Can I return a partially used product?" The answer depends on tone, policy nuance, completeness, and empathy.*
>
> *Welcome to the hardest unsolved problem in GenAI engineering.*

Let's see this problem in action with a concrete example.

In [5]:
import truststore
truststore.inject_into_ssl()
from openai import OpenAI
client = OpenAI(api_key=api_key)



# Our scenario: TechMart customer support chatbot
question = "What's your return policy for electronics?"

expected_answer = (
    "You can return most electronics within 30 days of purchase for a full refund. "
    "Items must be in original packaging with all accessories included. "
    "Opened software and digital downloads are non-refundable. "
    "For defective items, we offer a 90-day exchange warranty."
)

# Let's generate a chatbot response using OpenAI
response = client.responses.create(
    model="gpt-5-nano",
    input=[
        {"role": "system", "content": (
            "You are a helpful customer support agent for TechMart electronics store. "
            "TechMart's return policy: 30-day returns for electronics in original packaging "
            "with accessories. Opened software/digital downloads non-refundable. "
            "90-day exchange warranty for defective items."
        )},
        {"role": "user", "content": question}
    ]
)


actual_answer = response.output_text
print("📋 QUESTION:", question)
print()
print("✅ EXPECTED ANSWER:")
print(expected_answer)
print()
print("🤖 CHATBOT ANSWER:")
print(actual_answer)

📋 QUESTION: What's your return policy for electronics?

✅ EXPECTED ANSWER:
You can return most electronics within 30 days of purchase for a full refund. Items must be in original packaging with all accessories included. Opened software and digital downloads are non-refundable. For defective items, we offer a 90-day exchange warranty.

🤖 CHATBOT ANSWER:
Here’s TechMart’s return policy for electronics:

- 30-day return window from purchase date.
- Items must be in their original packaging and include all accessories.
- Opened software and digital downloads are non-refundable.
- 90-day exchange warranty for defective items (exchange rather than refund if item is defective).

If you’d like, I can help start a return or guide you through an exchange for a defective item. Please share your order number and the item.


In [6]:
misleading_answer = (
    "You can return most electronics within 30 days of purchase for a full refund. "
    "Items must be in original packaging with all accessories included. "
    "We also offer FREE LIFETIME WARRANTY on everything and PRICE MATCHING "
    "against any competitor!"  
)

1. Deterministic checks
2. LLM as Judge

# Deterministic Checks on every commit

In [8]:

# These are simple but catch real problems in production!

print("Actual answer: ", actual_answer)

def evaluate_heuristics(response: str) -> dict:
    """Basic code-based checks for a customer support chatbot."""
    checks = {}


        # Length check — too short = probably unhelpful, too long = overwhelming
    word_count = len(response.split())
    checks["appropriate_length"] = 20 <= word_count <= 300
    checks["word_count"] = word_count


    # Contains required elements
    checks["has_greeting_or_direct_answer"] = not response.startswith("I don't")


    checks["no_competitor_mentions"] = not any(
        comp in response.lower() for comp in ["amazon", "bestbuy", "best buy", "walmart"]
    )

      # Safety checks
    checks["no_profanity"] = not any(
        word in response.lower() for word in ["damn", "hell", "stupid"]
    )

        # Format check — should not contain raw code or system prompts
    checks["no_system_prompt_leak"] = "system:" not in response.lower()


    checks["no_raw_json"] = not response.strip().startswith("{")

    return checks



results = evaluate_heuristics(actual_answer)
for check, passed in results.items():
    status = "✅" if (passed if isinstance(passed, bool) else True) else "❌"
    print(f"  {status} {check}: {passed}")





Actual answer:  Here’s TechMart’s return policy for electronics:

- 30-day return window from purchase date.
- Items must be in their original packaging and include all accessories.
- Opened software and digital downloads are non-refundable.
- 90-day exchange warranty for defective items (exchange rather than refund if item is defective).

If you’d like, I can help start a return or guide you through an exchange for a defective item. Please share your order number and the item.
  ✅ appropriate_length: True
  ✅ word_count: 75
  ✅ has_greeting_or_direct_answer: True
  ✅ no_competitor_mentions: True
  ✅ no_profanity: True
  ✅ no_system_prompt_leak: True
  ✅ no_raw_json: True


# LLM as Judge - Game Changer

In [15]:
# we will create one from scratch. 


def llm_judge(question, response, criteria, model="gpt-5-nano"):
    """A simple LLM-as-a-Judge implementation from scratch."""

    judge_prompt = f"""You are an expert evaluator for a customer support chatbot.

    Evaluate the following response on this criteria: {criteria}

    USER QUESTION: {question}
    CHATBOT RESPONSE: {response}

    Score from 1-5 where:
    1 = Completely fails the criteria
    2 = Mostly fails with minor positives
    3 = Partially meets criteria
    4 = Mostly meets criteria with minor issues
    5 = Fully meets criteria

    NOTE: Attaching some policy about techmart

    "TechMart's return policy: 30-day returns for electronics in original packaging "
    "with accessories. Opened software/digital downloads non-refundable. "
    "90-day exchange warranty for defective items."

    Respond in this exact JSON format:
    {{"score": <int>, "reason": "<brief explanation>"}}"""

    result = client.responses.create(
        model=model,
        input=[{"role": "user", "content": judge_prompt}]
    )

    text = result.output_text.strip()
    return json.loads(text)
 

criteria_list = {
    "Accuracy": "Is the response factually correct based on TechMart's return policy?",
    "Helpfulness": "Does the response fully address the user's question in a helpful way?",
    "Tone": "Is the tone professional, friendly, and empathetic?",
    "Completeness": "Does the response cover all relevant aspects (timeframe, conditions, exceptions)?"
}


print("question: ", question)
print("response: ", actual_answer)

question:  What's your return policy for electronics?
response:  Here’s TechMart’s return policy for electronics:

- 30-day return window from purchase date.
- Items must be in their original packaging and include all accessories.
- Opened software and digital downloads are non-refundable.
- 90-day exchange warranty for defective items (exchange rather than refund if item is defective).

If you’d like, I can help start a return or guide you through an exchange for a defective item. Please share your order number and the item.


In [16]:
scores = {}
for name, criteria in criteria_list.items():
    result = llm_judge(question, actual_answer, criteria)
    scores[name] = result
    print(f"  {name}: {'⭐' * result['score']}{'☆' * (5-result['score'])} ({result['score']}/5)")
    print(f"    → {result['reason']}")
    print()

  Accuracy: ⭐⭐⭐⭐⭐ (5/5)
    → The response accurately reflects TechMart's stated electronics return policy: 30-day return window, original packaging with accessories, non-refundable for opened software/digital downloads, and a 90-day exchange warranty for defective items; phrasing and scope match policy.

  Helpfulness: ⭐⭐⭐⭐☆ (4/5)
    → The response clearly states TechMart's electronics return policy (30-day window, original packaging and all accessories, non-refundable opened software/digital downloads) and notes a 90-day exchange warranty for defective items. It also offers to start a return or guide an exchange and asks for order number. A stray 'NOTE' line and lack of explicit clarification on refunds versus exchanges and where/how to return slightly detract from completeness.

  Tone: ⭐⭐⭐⭐☆ (4/5)
    → Tone is professional and friendly; it clearly communicates the policy and offers help. It could be more empathetic by including a brief expression of understanding for the customer

In [17]:
avg_score = sum(s['score'] for s in scores.values()) / len(scores)
print(f"📊 Average Score: {avg_score:.1f}/5")

📊 Average Score: 4.2/5


In [18]:
scores = {}
for name, criteria in criteria_list.items():
    result = llm_judge(question, misleading_answer, criteria)
    scores[name] = result
    print(f"  {name}: {'⭐' * result['score']}{'☆' * (5-result['score'])} ({result['score']}/5)")
    print(f"    → {result['reason']}")
    print()

avg_score = sum(s['score'] for s in scores.values()) / len(scores)
print(f"📊 Average Score: {avg_score:.1f}/5")

  Accuracy: ⭐⭐⭐☆☆ (3/5)
    → TechMart policy specifies 30-day returns for electronics in original packaging with accessories; the response correctly states 30 days and packaging but adds unsupported claims (free lifetime warranty and price matching) and contradicts the policy's 90-day exchange warranty for defective items; it also omits policy detail about non-refundable software/digital downloads.

  Helpfulness: ⭐⭐⭐☆☆ (3/5)
    → Gives the core policy (30-day returns with original packaging/accessories) but includes unsupported claims (lifetime warranty, price matching) and omits important details like exceptions (opened software/digital downloads non-refundable) and whether refunds or exchanges apply.

  Tone: ⭐⭐⭐☆☆ (3/5)
    → The tone is professional and informative, but it lacks warmth and empathy. It provides policy details, but uses promotional language (e.g., 'FREE LIFETIME WARRANTY' and 'PRICE MATCHING') and could be more customer-friendly with a hint of empathy and clearer 

## G-EVAL

In [20]:

# G-Eval with DeepEval — Custom metrics in plain English

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel


judge_cheap = GPTModel(model="gpt-5-nano", api_key=api_key)  # A smaller, cheaper model for evaluation


# Metric 1: Customer Empathy (no expected output needed!)
empathy_metric = GEval(
    name="Customer Empathy",
    criteria="Evaluate whether the response demonstrates empathy and a customer-first attitude. The tone should be warm, professional, and make the customer feel valued.",
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    threshold=0.6,
	model=judge_cheap
)

to_test = LLMTestCase(
    input=question,
    actual_output=actual_answer)

empathy_metric.measure(to_test)




0.8

In [21]:

print(f"\n✅ Customer Empathy: {empathy_metric.score:.2f} (threshold: {empathy_metric.threshold})")
print(f"   Passed: {'✅ Yes' if empathy_metric.is_successful() else '❌ No'}")
print(f"   Reason: {empathy_metric.reason}")



✅ Customer Empathy: 0.80 (threshold: 0.6)
   Passed: ✅ Yes
   Reason: Strengths: The response directly answers the user's electronics return question, clearly stating a 30-day window, packaging and accessories requirement, non-refundable opened software, and a 90-day defect exchange, and it offers concrete next steps (assist with a return or exchange and requests order number). This demonstrates a customer-first posture. Shortcoming: It does not explicitly acknowledge the customer’s feelings or frustration, and could include a brief empathetic line to reinforce care.


In [22]:
# Metric 2: Factual Accuracy
accuracy_metric = GEval(
    name="Factual Accuracy",
    evaluation_steps=[
        "Compare each factual claim in the actual output against the expected output",
        "Check for any fabricated information not present in the expected output",
        "Penalize hallucinated policies, warranties, or offers",
        "Minor wording differences are acceptable if the facts are correct"
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    threshold=0.7,  # Score must be >= 0.7 to pass,
	model=judge_cheap
)

to_test = LLMTestCase(
    input=question,
    actual_output=actual_answer,
    expected_output=expected_answer
)

accuracy_metric.measure(to_test)
print(f"\n✅ Factual Accuracy: {accuracy_metric.score:.2f} (threshold: {accuracy_metric.threshold})")
print(f"   Passed: {'✅ Yes' if accuracy_metric.is_successful() else '❌ No'}")
print(f"   Reason: {accuracy_metric.reason}")



✅ Factual Accuracy: 0.50 (threshold: 0.7)
   Passed: ❌ No
   Reason: The actual output largely mirrors the policy (30-day window, original packaging, non-refundable software, 90-day exchange). However, it uses a generic '30-day return window' without stating a refund, while the expected specifies a 30-day window for a full refund. It also includes an extra closing offer to start a return, which is not in the expected output. These mismatches reduce alignment.


In [23]:
# --- Run G-Eval on the HALLUCINATED response ---
bad_test = LLMTestCase(
    input=question,
    actual_output=misleading_answer,
    expected_output=expected_answer
)

print("🚨 G-EVAL: Scoring the HALLUCINATED chatbot response")
print("=" * 60)

accuracy_metric.measure(bad_test)
print(f"\n❌ Factual Accuracy: {accuracy_metric.score:.2f} (threshold: {accuracy_metric.threshold})")
print(f"   Passed: {'✅ Yes' if accuracy_metric.is_successful() else '❌ No'}")
print(f"   Reason: {accuracy_metric.reason}")

empathy_metric.measure(bad_test)
print(f"\n🤔 Customer Empathy: {empathy_metric.score:.2f} (threshold: {empathy_metric.threshold})")
print(f"   Passed: {'✅ Yes' if empathy_metric.is_successful() else '❌ No'}")
print(f"   Reason: {empathy_metric.reason}")

print()
print("💡 NOTICE: G-Eval catches the hallucination on accuracy BUT may score")
print("   empathy higher (because the fake promises sound helpful!).")
print("   This is why you need MULTIPLE metrics — no single metric tells the whole story.")

🚨 G-EVAL: Scoring the HALLUCINATED chatbot response



❌ Factual Accuracy: 0.30 (threshold: 0.7)
   Passed: ❌ No
   Reason: The actual output correctly repeats the 30-day refund and original-packaging requirement, but adds unlisted policies (free lifetime warranty and price matching) not in the expected output, and omits the expected items (non-refundable opened software/digital downloads and a 90-day warranty for defects). This shows clear fabrication and omissions, reducing alignment.



🤔 Customer Empathy: 0.40 (threshold: 0.6)
   Passed: ❌ No
   Reason: Directly addresses the return policy with a 30-day window and packaging requirement, but lacks warmth/empathy, provides no clear steps to start a return, and adds extraneous offers (lifetime warranty, price matching) beyond the request, which could confuse the customer.

💡 NOTICE: G-Eval catches the hallucination on accuracy BUT may score
   empathy higher (because the fake promises sound helpful!).
   This is why you need MULTIPLE metrics — no single metric tells the whole story.


In [25]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase

question = "Is replacement free after warranty?"
expected = "No. After warranty expiry, replacement is not free; offer paid repair."

candidates = [
    "Yes, free replacement for 2 years.",
    "No, after warranty it’s paid repair; I can share pricing options.",
    "Not sure.",
]

test_cases = [
    LLMTestCase(input=question, actual_output=a, expected_output=expected)
    for a in candidates
]

evaluate(test_cases=test_cases, metrics=[accuracy_metric, empathy_metric])

✨ You're running DeepEval's latest Factual Accuracy [GEval] Metric! (using gpt-5-nano, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Customer Empathy [GEval] Metric! (using gpt-5-nano, strict=False, 
async_mode=True)...



Metrics Summary

  - ❌ Factual Accuracy [GEval] (score: 0.0, threshold: 0.7, strict: False, evaluation model: gpt-5-nano, reason: The actual output 'Not sure.' does not match the expected definitive policy: it fails to state that replacement is not free after warranty expiry and omits the required 'paid repair' detail., error: None)
  - ❌ Customer Empathy [GEval] (score: 0.0, threshold: 0.6, strict: False, evaluation model: gpt-5-nano, reason: The actual output 'Not sure.' does not address the user's question about whether replacement is free after warranty. It lacks empathy, warmth, and any actionable guidance or next steps. It also fails to acknowledge the user's potential concern or offer to check policy or provide a clear answer; a better reply would confirm the warranty terms or offer to check the specific order and outline steps to resolve., error: None)

For test case:

  - input: Is replacement free after warranty?
  - actual output: Not sure.
  - expected output: No. After w

⚠ WARNING: No hyperparameters logged.
» ]8;id=338099;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 13.37s | token cost: 0.0029723000000000006 USD)
» Test Results (3 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 3

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

EvaluationResult(test_results=[TestResult(name='test_case_2', success=False, metrics_data=[MetricData(name='Factual Accuracy [GEval]', threshold=0.7, success=False, score=0.0, reason="The actual output 'Not sure.' does not match the expected definitive policy: it fails to state that replacement is not free after warranty expiry and omits the required 'paid repair' detail.", strict_mode=False, evaluation_model='gpt-5-nano', error=None, evaluation_cost=0.0004512, verbose_logs='Criteria:\nNone \n \nEvaluation Steps:\n[\n    "Compare each factual claim in the actual output against the expected output",\n    "Check for any fabricated information not present in the expected output",\n    "Penalize hallucinated policies, warranties, or offers",\n    "Minor wording differences are acceptable if the facts are correct"\n] \n \nRubric:\nNone \n \nScore: 0.0'), MetricData(name='Customer Empathy [GEval]', threshold=0.6, success=False, score=0.0, reason="The actual output 'Not sure.' does not addres

# How to Evaluate RAG Based LLM!?